# 📚 Book Recommendation System

**Dataset:** [Book-Crossing Dataset](https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset)  
**Approach:** Collaborative Filtering · K-Nearest Neighbours · Cosine Similarity

## 1. Introduction

A **recommendation system** is an information-filtering tool that predicts a user's preference for an item and surfaces the most relevant suggestions. Three dominant paradigms exist:

| Type | How it works | Strength | Weakness |
|------|-------------|----------|----------|
| **Content-based** | Recommends items similar *in attributes* to what the user liked (e.g. same genre, author). | Works for new users with item metadata. | Cannot discover cross-genre serendipity. |
| **Collaborative Filtering** | Recommends items liked by *users with similar taste*. Ignores item attributes entirely. | Captures latent patterns humans don't articulate. | Cold-start problem for new users/items. |
| **Hybrid** | Combines both signals (e.g. Netflix, Spotify). | Best accuracy in practice. | Higher engineering complexity. |

This notebook implements **item-based collaborative filtering**: we build a user–book rating matrix and use **K-Nearest Neighbours** with **cosine similarity** to find books that attract the same audience.

In [ ]:
import os
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

---
## 2. Data Loading

The dataset ships as three CSV files in the `data/` directory:

* **Books.csv** — ISBN, title, author, year, publisher, cover image URLs  
* **Ratings.csv** — `(User-ID, ISBN, Book-Rating)` triples (0 = implicit, 1-10 = explicit)  
* **Users.csv** — anonymised user demographics

In [ ]:
DATA_DIR = 'data'

books   = pd.read_csv(os.path.join(DATA_DIR, 'Books.csv'),   encoding='latin-1', on_bad_lines='skip')
ratings = pd.read_csv(os.path.join(DATA_DIR, 'Ratings.csv'), encoding='latin-1')
users   = pd.read_csv(os.path.join(DATA_DIR, 'Users.csv'),   encoding='latin-1')

print(f'Books   : {books.shape[0]:>8,} rows  x  {books.shape[1]} columns')
print(f'Ratings : {ratings.shape[0]:>8,} rows  x  {ratings.shape[1]} columns')
print(f'Users   : {users.shape[0]:>8,} rows  x  {users.shape[1]} columns')

In [ ]:
from IPython.display import display

print('=== Books (first 3 rows) ===')
display(books[['ISBN','Book-Title','Book-Author','Year-Of-Publication','Publisher']].head(3))

print('=== Ratings (first 3 rows) ===')
display(ratings.head(3))

print('=== Users (first 3 rows) ===')
display(users.head(3))

---
## 3. Analysis & Exploratory Data Analysis

Before building the model we inspect data quality, the rating scale, and the
activity distribution across users and books.

In [ ]:
def missing_summary(df, name):
    mv = df.isnull().sum()
    mv = mv[mv > 0]
    if mv.empty:
        print(f'{name}: no missing values')
    else:
        pct = (mv / len(df) * 100).round(2)
        print(f'{name} — missing values:')
        print(pd.concat([mv.rename('count'), pct.rename('pct%')], axis=1).to_string())
    print()

missing_summary(books,   'Books')
missing_summary(ratings, 'Ratings')
missing_summary(users,   'Users')

In [ ]:
explicit = ratings[ratings['Book-Rating'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All ratings including implicit zeros
ratings['Book-Rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('All Ratings  (0 = implicit interaction)', fontsize=12)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Explicit ratings only
explicit['Book-Rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title(f'Explicit Ratings 1-10  ({len(explicit):,} total)', fontsize=12)
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Total ratings      : {len(ratings):>10,}')
print(f'Explicit (1-10)    : {len(explicit):>10,}  ({100*len(explicit)/len(ratings):.1f}%)')
print(f'Implicit (0)       : {len(ratings)-len(explicit):>10,}  ({100*(1-len(explicit)/len(ratings)):.1f}%)')

In [ ]:
ratings_per_user = ratings.groupby('User-ID')['Book-Rating'].count()
ratings_per_book = ratings.groupby('ISBN')['Book-Rating'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(ratings_per_user[ratings_per_user <= 500], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Ratings per User (capped at 500)', fontsize=12)
axes[0].set_xlabel('Number of ratings')
axes[0].set_ylabel('Number of users')

axes[1].hist(ratings_per_book[ratings_per_book <= 200], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Ratings per Book (capped at 200)', fontsize=12)
axes[1].set_xlabel('Number of ratings')
axes[1].set_ylabel('Number of books')

plt.tight_layout()
plt.show()

print('=== User activity ===')
print(f'  Average ratings per user : {ratings_per_user.mean():.1f}')
print(f'  Median ratings per user  : {ratings_per_user.median():.0f}')
print(f'  Max ratings by one user  : {ratings_per_user.max():,}')
print()
print('=== Book popularity ===')
print(f'  Average ratings per book : {ratings_per_book.mean():.1f}')
print(f'  Median ratings per book  : {ratings_per_book.median():.0f}')
print(f'  Max ratings for one book : {ratings_per_book.max():,}')

---
## 4. Preprocessing

The dataset is highly sparse and dominated by low-activity users and obscure books.
Keeping them would produce meaningless neighbours. We apply two threshold filters:

1. **Active users** — at least `MIN_USER_RATINGS` (200) ratings.  
   These users have broad enough taste for a meaningful cosine vector.
2. **Popular books** — at least `MIN_BOOK_RATINGS` (50) ratings.  
   Ensures the book's rating vector is dense enough for reliable similarity.

After filtering we merge with `books` to resolve ISBNs to titles, deduplicate,
and build the pivot-table matrix used by the model.

In [ ]:
MIN_USER_RATINGS = 200
MIN_BOOK_RATINGS = 50

# --- Filter active users ---
user_counts  = ratings['User-ID'].value_counts()
active_users = user_counts[user_counts >= MIN_USER_RATINGS].index

# --- Filter popular books ---
book_counts   = ratings['ISBN'].value_counts()
popular_isbns = book_counts[book_counts >= MIN_BOOK_RATINGS].index

filtered_ratings = ratings[
    ratings['User-ID'].isin(active_users) &
    ratings['ISBN'].isin(popular_isbns)
]

print(f'Active users   (>= {MIN_USER_RATINGS} ratings) : {len(active_users):,}  /  {ratings["User-ID"].nunique():,} total')
print(f'Popular ISBNs  (>= {MIN_BOOK_RATINGS} ratings) : {len(popular_isbns):,}  /  {ratings["ISBN"].nunique():,} total')
print(f'Filtered ratings                       : {len(filtered_ratings):,}  /  {len(ratings):,} total')

In [ ]:
# Merge filtered ratings with book metadata (title + author)
merged = filtered_ratings.merge(
    books[['ISBN', 'Book-Title', 'Book-Author']],
    on='ISBN',
    how='left'
)

# Drop rows where the title couldn't be resolved
merged = merged.dropna(subset=['Book-Title'])

# A user may have rated the same title under different ISBNs — keep only one
merged = merged.drop_duplicates(['User-ID', 'Book-Title'])

print(f'Merged ratings  : {len(merged):,}')
print(f'Unique books    : {merged["Book-Title"].nunique():,}')
print(f'Unique users    : {merged["User-ID"].nunique():,}')
display(merged.head(5))

In [ ]:
# Build the book–user matrix: rows = books, columns = users, values = ratings
pivot_table = merged.pivot_table(
    index='Book-Title',
    columns='User-ID',
    values='Book-Rating',
    fill_value=0
)

n_books, n_users = pivot_table.shape
sparsity = 1 - len(merged) / (n_books * n_users)

print(f'Pivot table : {n_books:,} books  x  {n_users:,} users')
print(f'Sparsity    : {sparsity:.2%}  (fraction of cells that are zero)')
print(f'Memory      : {pivot_table.memory_usage().sum() / 1024**2:.1f} MB (dense)')
display(pivot_table.iloc[:5, :5])

---
## 5. Model Building — K-Nearest Neighbours

We use `sklearn.neighbors.NearestNeighbors` with **cosine distance**.

**Why cosine?**  
Two users who both rate a book 8 and 9 are more similar than one who rates 2 and 3,
even if the raw dot product is different. Cosine normalises for vector magnitude,
so it captures *relative preference patterns* rather than absolute rating levels.

**Why brute-force?**  
The matrix has ~700 books after filtering — exact brute-force search is fast at this
scale and avoids approximation errors from tree structures.

In [ ]:
# Convert to CSR sparse matrix — saves memory and speeds up cosine computations
sparse_matrix = csr_matrix(pivot_table.values)

model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model.fit(sparse_matrix)

print(f'NearestNeighbors fitted')
print(f'  Matrix shape : {sparse_matrix.shape}')
print(f'  Non-zero     : {sparse_matrix.nnz:,}  ({100*sparse_matrix.nnz/(n_books*n_users):.2f}% density)')

---
## 6. Inference — Recommendation Function

`get_recommendations` wraps `model.kneighbors()` with input validation and
returns a ranked list of similar books with their cosine similarity scores.

In [ ]:
def get_recommendations(book_name, model, pivot_table, n_recommendations=5):
    """
    Return the top-n books most similar to *book_name* using KNN cosine distance.

    Parameters
    ----------
    book_name : str
        Exact title that must appear in pivot_table.index.
    model : NearestNeighbors
        Fitted model.
    pivot_table : pd.DataFrame
        Book-user rating matrix (books as rows).
    n_recommendations : int
        How many neighbours to return (excluding the query book itself).

    Returns
    -------
    list[dict]  — keys: 'title', 'distance', 'similarity'

    Raises
    ------
    ValueError  if book_name is not in the dataset.
    """
    matches = np.where(pivot_table.index == book_name)[0]
    if len(matches) == 0:
        raise ValueError(f"Book not found in dataset: '{book_name}'")

    book_idx = matches[0]

    # kneighbors returns distances and indices; n_neighbors=n+1 because
    # the query book itself is always the first (distance=0) result.
    distances, indices = model.kneighbors(
        pivot_table.iloc[book_idx, :].values.reshape(1, -1),
        n_neighbors=n_recommendations + 1,
    )

    recommendations = [
        {
            'title':      pivot_table.index[indices[0][i]],
            'distance':   round(float(distances[0][i]), 4),
            'similarity': round(1.0 - float(distances[0][i]), 4),
        }
        for i in range(1, len(indices[0]))  # skip index 0 (the query book)
    ]
    return recommendations

In [ ]:
# Pick a well-known title; fall back to the first entry if absent
QUERY = 'The Da Vinci Code'
if QUERY not in pivot_table.index:
    QUERY = pivot_table.index[0]
    print(f'Fallback: using "{QUERY}"')

print(f'Query book : "{QUERY}"')
print('-' * 60)

recs = get_recommendations(QUERY, model, pivot_table, n_recommendations=5)

for rank, rec in enumerate(recs, start=1):
    title = rec['title']
    sim   = rec['similarity']
    dist  = rec['distance']
    print(f'{rank}.  {title:<50}  similarity={sim:.4f}  distance={dist:.4f}')

---
## 7. Saving Artefacts

Three objects are serialised with `pickle.dump()` into the `models/` directory:

| File | Contents | Used by |
|------|----------|---------|
| `model.pkl` | Fitted `NearestNeighbors` | Inference |
| `pivot_table.pkl` | Book–user matrix + book-title index | Inference |
| `books_data.pkl` | `DataFrame` mapping title → author | Streamlit UI |

In [ ]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# --- model (sklearn has no native save format, so pickle is appropriate here) ---
with open(os.path.join(MODELS_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump(model, f)

# --- pivot table (parquet is cross-version and ~5x smaller than pickle) ---
pivot_table.to_parquet(os.path.join(MODELS_DIR, 'pivot_table.parquet'))

# --- book metadata ---
book_metadata = (
    books[['ISBN', 'Book-Title', 'Book-Author']]
    .drop_duplicates('Book-Title')
    .set_index('Book-Title')
)
book_metadata = book_metadata[book_metadata.index.isin(pivot_table.index)]
book_metadata.to_parquet(os.path.join(MODELS_DIR, 'books_data.parquet'))

# --- report sizes ---
print('Artefacts written to models/')
for fname in ['model.pkl', 'pivot_table.parquet', 'books_data.parquet']:
    path = os.path.join(MODELS_DIR, fname)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {fname:<25}  {size_kb:>8,.0f} KB')

print()
print('Run: streamlit run app.py')